# Module 3 – Building Semantic Search from Scratch
### Course: OpenAI Embeddings API | Pluralsight

---
## What You'll Learn
- Similarity metrics: cosine similarity, dot product, Euclidean distance
- Building an in-memory search index with numpy
- Re-ranking strategies for better relevance
- Hybrid search: BM25 keyword scoring + semantic embeddings
- Evaluating search quality: precision@k, recall@k, MRR

---

In [ ]:
# ─ SETUP · 1 — Install dependencies ────────────────────────────────
# !pip install openai numpy scikit-learn rank-bm25

In [ ]:
# ─ SETUP · 2 — Imports & load data ─────────────────────────────────
import os
import getpass
import json
import numpy as np
from openai import OpenAI
from rank_bm25 import BM25Okapi

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key: ")
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

with open("../module2/yelp_embeddings.json") as f:
    embedded_reviews = json.load(f)

print(f"Loaded {len(embedded_reviews)} embedded reviews")
print(f"Vector dimensions: {len(embedded_reviews[0]['embedding'])}")

---
## 1: Similarity Metrics and Search Architecture

**The whole architecture, in four steps:**
1. Embed the corpus (done back in Module 2)
2. Embed the query
3. Compute similarity between the query and every document
4. Return the top results

| Metric | Formula | Best when |
|---|---|---|
| **Cosine** | `dot(a, b) / (‖a‖ · ‖b‖)` | Normalized vectors — the default for text embeddings |
| **Dot product** | `dot(a, b)` | Magnitude carries meaning |
| **Euclidean** | `‖a − b‖` | Absolute distance matters (lower = more similar) |

In [ ]:
# ─ 1 · 1 — The three similarity metrics ───────────────────────
def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Direction-only similarity in [-1, 1]. Higher = more similar."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def dot_product(a: list[float], b: list[float]) -> float:
    """Like cosine but NOT normalized, so magnitude still counts."""
    return float(np.dot(np.array(a), np.array(b)))


def euclidean_distance(a: list[float], b: list[float]) -> float:
    """Straight-line distance. Lower = more similar."""
    return float(np.linalg.norm(np.array(a) - np.array(b)))

In [ ]:
# ─ 1 · 2 — The reviews we'll compare ──────────────────────────

# We hand-picked three reviews by scanning the Yelp corpus:
#   reviews 40 & 41 are the SAME diner (Gab n Eat); review 62 is a Chinese takeout.

## TODO: #1 - See all three reviews.

In [ ]:
# ─ 1 · 3 — Seeing it on real reviews ──────────────────────────

review_a = embedded_reviews[40]  # Gab n Eat diner - happy review
review_b = embedded_reviews[41]  # Gab n Eat diner - another happy review (SAME place)
review_c = embedded_reviews[62]  # a Chinese-takeout review (different place)

## TODO: #2 - Compare the three reviews with each other.


### 1 · 4 — Why we vectorize

`cosine_similarity` above compares **one pair** at a time. To search, we need the query
scored against **all 500** reviews — and in production, millions. A Python `for` loop is
far too slow, so we *vectorize*: normalize everything once, then let a single matrix
multiply score every document at the same time.

In [ ]:
# ─ 1 · 5 — The vectorized engine ──────────────────────────────
def cosine_similarity_matrix(query_vec: np.ndarray, corpus_matrix: np.ndarray) -> np.ndarray:
    """
    Cosine similarity between ONE query vector and EVERY corpus vector.
        query_vec:     shape (dim,)
        corpus_matrix: shape (N, dim)
        returns:       shape (N,)   one score per document
    """
    # 1. Make the query a unit vector (length 1)
    query_norm = query_vec / np.linalg.norm(query_vec)
    # 2. Make every corpus row a unit vector
    corpus_norms = np.linalg.norm(corpus_matrix, axis=1, keepdims=True)  # (N, 1)
    corpus_normalized = corpus_matrix / corpus_norms                     # (N, dim)
    # 3. Dot product of unit vectors IS cosine similarity - all N at once
    return corpus_normalized @ query_norm                               # (N,)


## TODO: #3 - Sanity check: the vectorized version must agree with the per-pair version.

---
## 2: Building and Querying a Search Index

In [ ]:
# ─ 2 · 1 — Build the index ────────────────────────────────────
# The "index" is just two parallel structures: a matrix of vectors + a list of metadata.

corpus_matrix = np.array([r["embedding"] for r in embedded_reviews])  # (N, 1536)
metadata = [{"id": r["id"], "text": r["text"], "stars": r["stars"]} for r in embedded_reviews]

print(f"Index built: {corpus_matrix.shape}  ({corpus_matrix.shape[0]} reviews x {corpus_matrix.shape[1]} dims)")
print("Row 0 metadata:", {"id": metadata[0]["id"], "stars": metadata[0]["stars"], "text": metadata[0]["text"][:50] + "..."})

In [ ]:
# ─ 2 · 2 — The search function ────────────────────────────────
def semantic_search(query: str, top_k: int = 5) -> list[dict]:
    """Return the top_k most relevant reviews for a query."""
    # Step 1: Embed the query - SAME model family as the corpus, or the spaces won't match
    response = client.embeddings.create(input=query, model="text-embedding-3-small")
    query_vec = np.array(response.data[0].embedding)

    # Step 2: Vectorized cosine similarity against all docs
    scores = cosine_similarity_matrix(query_vec, corpus_matrix)

    # Step 3: Take the indices of the highest-scoring docs
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {"rank": i + 1, "idx": int(idx), "score": float(scores[idx]),
         "stars": metadata[idx]["stars"], "text": metadata[idx]["text"]}
        for i, idx in enumerate(top_indices)
    ]

print("semantic_search defined.")

In [ ]:
# ─ 2 · 3 — Search that understands meaning ────────────────────
# Semantic search finds relevant reviews even when the query words never appear in them.

q1 = "romantic dinner spot with live music"
q2 = "quick healthy lunch near downtown"

## TODO: #4 - Run Semantic Search with these two queries. 

In [ ]:
# ─ 2 · 4 — The honest limitation ──────────────────────────────
# The honest limitation: search ALWAYS returns its nearest guess, even with no real match.
# There is no ramen review in our 500 - watch the top scores drop as a "nothing fits" signal.

query = "authentic ramen with rich broth"
print(f"Query: '{query}'  (no ramen exists in this corpus)\n{'-'*60}")

for r in semantic_search(query, top_k=3):
    print(f"  #{r['rank']} [score={r['score']:.4f}, {r['stars']}*] {r['text'][:75]}...")

In [ ]:
# ─ 2 · 5 — Re-ranking (define) ────────────────────────────────
def semantic_search_with_reranking(query, candidate_k=20, final_k=5, star_boost=0.05):
    """Two-stage: retrieve a BROAD candidate set cheaply, then re-rank a NARROW set."""

    candidates = semantic_search(query, top_k=candidate_k)        # retrieve broad

    ## TODO: #5 - re-rank the candidates.

print("semantic_search_with_reranking defined.")

In [ ]:
# ─ 2 · 6 — Re-ranking (compare) ───────────────────────────────
query = "cozy place for a quiet dinner"
print(f"Query: '{query}'\n\n--- BASE SEARCH (similarity only) ---")
for r in semantic_search(query, top_k=5):
    print(f"  [{r['stars']}*, score={r['score']:.4f}] {r['text'][:65]}...")

print("\n--- RE-RANKED (similarity + star-quality boost) ---")
for r in semantic_search_with_reranking(query, final_k=5):
    print(f"  [{r['stars']}*, final={r['reranked_score']:.4f}] {r['text'][:65]}...")

---
## 3: Hybrid Search and Quality Evaluation

In [ ]:
# ─ 3 · 1 — Build the BM25 keyword index ───────────────────────
# BM25 works on TOKENS, not vectors: pure literal word-overlap scoring.
print("Building BM25 keyword index...")

## TODO: #6 - Build the BM25 index on the review texts.

In [ ]:
# ─ 3 · 2 — Keyword results ────────────────────────────────────
import string

def bm25_search(query: str, top_k: int = 20) -> list[dict]:
    """Pure keyword search. Rare words weigh more than common ones."""
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{"rank": i + 1, "idx": int(idx), "score": float(scores[idx])}
            for i, idx in enumerate(top_indices)]


def matched_terms(text: str, query: str) -> list[str]:
    """Which query words literally appear in the review — i.e. WHY BM25 matched it."""
    q = set(query.lower().split())
    words = {w.lower().strip(string.punctuation) for w in text.split()}
    return sorted(q & words)


def snippet(text: str, query: str, width: int = 64) -> str:
    """A window centered on the first matching word, so the match is visible on screen."""
    low = text.lower()
    positions = [low.find(w) for w in query.lower().split() if w in low]
    pos = min(positions) if positions else 0
    start = max(0, pos - width // 3)
    window = text[start:start + width].replace("\n", " ")
    return ("…" if start else "") + window + "…"


# Show WHY each result matched: the literal query words it contains, and a snippet on the match.
query = "pizza near downtown"
print(f"Top BM25 keyword matches for '{query}':\n")
for r in bm25_search(query, top_k=3):
    text = metadata[r["idx"]]["text"]
    print(f"  [bm25={r['score']:.2f}]  matched {matched_terms(text, query)}")
    print(f"      {snippet(text, query)}\n")


In [ ]:
# ─ 3 · 3 — Reciprocal Rank Fusion ─────────────────────────────
def reciprocal_rank_fusion(semantic_results, bm25_results, k: int = 60, final_k: int = 5):
    """Combine two rankings using RANK only - this sidesteps the score-scale mismatch."""
    rrf_scores = {}
    for rank, result in enumerate(semantic_results, 1):              # contribution from semantic
        rrf_scores[result["idx"]] = rrf_scores.get(result["idx"], 0) + 1 / (k + rank)
    for rank, result in enumerate(bm25_results, 1):                  # contribution from BM25
        rrf_scores[result["idx"]] = rrf_scores.get(result["idx"], 0) + 1 / (k + rank)

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [{"rank": i + 1, "idx": idx, "rrf_score": score,
             "stars": metadata[idx]["stars"], "text": metadata[idx]["text"]}
            for i, (idx, score) in enumerate(ranked[:final_k])]

print("reciprocal_rank_fusion defined.")

In [ ]:
# ─ 3 · 4 — Hybrid search in action ────────────────────────────
def hybrid_search(query: str, top_k: int = 5) -> list[dict]:
    """Fuse semantic (meaning) and BM25 (exact keywords) via Reciprocal Rank Fusion."""

    ## TODO: #7 - Run both semantic and BM25 search, then fuse with RRF.

In [ ]:
# ─ 3 · 5 — Metadata filtering ─────────────────────────────────
def semantic_search_filtered(query: str, min_stars: int = 4, top_k: int = 5) -> list[dict]:
    """Metadata constraint. Over-retrieve, THEN filter, THEN truncate - or you starve the list."""

    candidates = semantic_search(query, top_k=top_k * 3)
    ## TODO: #8 - Filter the candidates by star rating, then take the top_k of the survivors.

### 3 · 6 — Why evaluate
---
### Evaluating search quality

You can't tune what you can't measure. Evaluation needs **ground truth**: for each test
query, the set of reviews a human judged genuinely relevant. That is exactly what
`relevant_indices` is — a hand-labeled answer key, stored as review **ids**.

How do you build it across 500 reviews? You surface on-topic candidates with a quick
keyword scan, then *read and confirm* them. The next cell shows that derivation, so the
numbers in `eval_set` are real, not guessed.

In [ ]:
# ─ 3 · 7 — Where the ground truth comes from ──────────────────
# HOW WE BUILT THE GROUND TRUTH
# relevant_indices = the review ids a human confirmed as truly on-topic for a query.
# Step 1: surface candidates with a keyword scan.  Step 2 (offline): read them and keep the good ids.
def show_candidates(*keywords, limit=8):
    shown = 0
    for r in embedded_reviews:
        if any(k in r["text"].lower() for k in keywords):
            print(f"  id={r['id']:>3}  {r['stars']}*  {r['text'][:62]}...")
            shown += 1
            if shown == limit:
                break

print("Candidates for 'pizza':")
show_candidates("pizza", "pizzeria")
print("\nCandidates for 'classic diner breakfast':")
show_candidates("diner", "breakfast", "brunch")

In [ ]:
# ─ 3 · 8 — The answer key ─────────────────────────────────────
# The answer key, confirmed by reading the candidates above.
# NOTE: in this dataset a review's id equals its row position, so these ids line up
# directly with the idx that semantic_search returns.
eval_set = [
    {"query": "pizza",                   "relevant_indices": [52, 69, 71, 74]},
    {"query": "classic diner breakfast", "relevant_indices": [34, 38, 40, 41, 42]},
    {"query": "chinese takeout",         "relevant_indices": [60, 62, 63]},
    {"query": "great cheeseburger",      "relevant_indices": [19, 23]},
    {"query": "bar with good beer",      "relevant_indices": [20, 21, 35]},
]
print(f"{len(eval_set)} labeled queries ready.")

In [ ]:
# ─ 3 · 9 — The three eval metrics ─────────────────────────────
def precision_at_k(retrieved_indices: list[int], relevant_indices: list[int], k: int) -> float:
    """Of the top-k returned, what fraction are relevant? (Did we waste the user's attention?)"""
    top_k = set(retrieved_indices[:k])
    return len(top_k & set(relevant_indices)) / k


def recall_at_k(retrieved_indices: list[int], relevant_indices: list[int], k: int) -> float:
    """Of ALL relevant items, what fraction did we surface in the top-k? (Did we miss things?)"""
    relevant = set(relevant_indices)
    if not relevant:
        return 0.0
    return len(set(retrieved_indices[:k]) & relevant) / len(relevant)


def mean_reciprocal_rank(retrieved_indices: list[int], relevant_indices: list[int]) -> float:
    """1 / rank of the FIRST relevant result. Rewards putting something good near the top."""
    relevant = set(relevant_indices)
    for rank, idx in enumerate(retrieved_indices, 1):
        if idx in relevant:
            return 1.0 / rank
    return 0.0

print("Metrics defined.")

In [ ]:
# ─ 3 · 10 — Reading the results & improving ───────────────────
import pandas as pd

K = 5
rows = []
for item in eval_set:
    retrieved = [r["idx"] for r in semantic_search(item["query"], top_k=K)]
    rows.append({
        "query":   item["query"],
        "labeled": len(item["relevant_indices"]),          # how many we marked relevant
        f"P@{K}":  precision_at_k(retrieved, item["relevant_indices"], K),
        f"R@{K}":  recall_at_k(retrieved, item["relevant_indices"], K),
        "MRR":     mean_reciprocal_rank(retrieved, item["relevant_indices"]),
    })

results = pd.DataFrame(rows).round(2)

# Append a MEAN row for the headline numbers
mean_row = {
    "query": "MEAN", "labeled": "",
    f"P@{K}": round(results[f"P@{K}"].mean(), 3),
    f"R@{K}": round(results[f"R@{K}"].mean(), 3),
    "MRR":    round(results["MRR"].mean(), 3),
}
results = pd.concat([results, pd.DataFrame([mean_row])], ignore_index=True)

print(f"semantic_search evaluation  (k={K})")
results
